# Olist E-commerce Data Audit

## Purpose

This notebook performs an initial audit of the Olist e-commerce dataset before formal analysis.

The audit will:

- confirm that all expected CSV files are available;
- inspect the size and columns of each table;
- identify missing values;
- understand the relationships between tables;
- determine which tables are required to study delivery delays and customer satisfaction.

## Main research question

How do delivery delays affect customer satisfaction?

## Division of work

This notebook covers the initial structural audit. A separate statistical audit will later examine duplicates, date logic, outliers, delay distributions and review-score differences.

In [1]:
from pathlib import Path
import pandas as pd
import duckdb

project_root = Path.cwd()

if not (project_root / "data" / "raw").exists():
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"
csv_files = sorted(raw_data_dir.glob("*.csv"))

print("Raw data folder:", raw_data_dir.resolve())
print("Number of CSV files:", len(csv_files))

for file in csv_files:
    print("-", file.name)

Raw data folder: F:\Personal_interesting_projects\ecommerce-delivery-analysis\Main file\ecommerce-delivery-analysis\data\raw
Number of CSV files: 9
- olist_customers_dataset.csv
- olist_geolocation_dataset.csv
- olist_order_items_dataset.csv
- olist_order_payments_dataset.csv
- olist_order_reviews_dataset.csv
- olist_orders_dataset.csv
- olist_products_dataset.csv
- olist_sellers_dataset.csv
- product_category_name_translation.csv


In [2]:
table_summary = []

for file in csv_files:
    row_count = duckdb.execute(
        "SELECT COUNT(*) FROM read_csv_auto(?)",
        [file.as_posix()]
    ).fetchone()[0]
    
    column_names = pd.read_csv(file, nrows=0).columns.tolist()
    
    table_summary.append({
        "table": file.stem,
        "rows": row_count,
        "columns": len(column_names),
        "size_mb": round(file.stat().st_size / (1024 ** 2), 2)
    })

table_summary = (
    pd.DataFrame(table_summary)
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)

table_summary

,table,rows,columns,size_mb
0,olist_geolocation_dataset,1000163,5,58.44
1,olist_order_items_dataset,112650,7,14.72
2,olist_order_payments_dataset,103886,5,5.51
3,olist_customers_dataset,99441,5,8.62
4,olist_orders_dataset,99441,8,16.84
5,olist_order_reviews_dataset,99224,7,13.78
6,olist_products_dataset,32951,9,2.27
7,olist_sellers_dataset,3095,4,0.17
8,product_category_name_translation,71,2,0.00


In [3]:
for file in csv_files:
    columns = pd.read_csv(file, nrows=0).columns.tolist()
    
    print(f"\n{file.stem} ({len(columns)} columns)")
    for column in columns:
        print(" -", column)


olist_customers_dataset (5 columns)
 - customer_id
 - customer_unique_id
 - customer_zip_code_prefix
 - customer_city
 - customer_state

olist_geolocation_dataset (5 columns)
 - geolocation_zip_code_prefix
 - geolocation_lat
 - geolocation_lng
 - geolocation_city
 - geolocation_state

olist_order_items_dataset (7 columns)
 - order_id
 - order_item_id
 - product_id
 - seller_id
 - shipping_limit_date
 - price
 - freight_value

olist_order_payments_dataset (5 columns)
 - order_id
 - payment_sequential
 - payment_type
 - payment_installments
 - payment_value

olist_order_reviews_dataset (7 columns)
 - review_id
 - order_id
 - review_score
 - review_comment_title
 - review_comment_message
 - review_creation_date
 - review_answer_timestamp

olist_orders_dataset (8 columns)
 - order_id
 - customer_id
 - order_status
 - order_purchase_timestamp
 - order_approved_at
 - order_delivered_carrier_date
 - order_delivered_customer_date
 - order_estimated_delivery_date

olist_products_dataset (9 col

In [4]:
missing_summary = []

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)
    missing_counts = df.isna().sum()
    
    for column, missing_count in missing_counts.items():
        if missing_count > 0:
            missing_summary.append({
                "table": file.stem,
                "column": column,
                "missing_values": int(missing_count),
                "missing_percent": round(missing_count / len(df) * 100, 2)
            })

missing_summary = (
    pd.DataFrame(missing_summary)
    .sort_values(
        ["missing_percent", "missing_values"],
        ascending=False
    )
    .reset_index(drop=True)
)

missing_summary

,table,column,missing_values,missing_percent
0,olist_order_reviews_dataset,review_comment_title,87656,88.34
1,olist_order_reviews_dataset,review_comment_message,58247,58.70
2,olist_orders_dataset,order_delivered_customer_date,2965,2.98
3,olist_products_dataset,product_category_name,610,1.85
4,olist_products_dataset,product_name_lenght,610,1.85
5,olist_products_dataset,product_description_lenght,610,1.85
6,olist_products_dataset,product_photos_qty,610,1.85
7,olist_orders_dataset,order_delivered_carrier_date,1783,1.79
8,olist_orders_dataset,order_approved_at,160,0.16
9,olist_products_dataset,product_weight_g,2,0.01


In [5]:
orders_file = raw_data_dir / "olist_orders_dataset.csv"

order_status_summary = duckdb.execute(
    """
    SELECT
        order_status,
        COUNT(*) AS order_count,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS percent
    FROM read_csv_auto(?)
    GROUP BY order_status
    ORDER BY order_count DESC
    """,
    [orders_file.as_posix()]
).df()

order_status_summary

,order_status,order_count,percent
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [6]:
reviews_file = raw_data_dir / "olist_order_reviews_dataset.csv"

analysis_coverage = duckdb.execute(
    """
    WITH reviewed_orders AS (
        SELECT DISTINCT order_id
        FROM read_csv_auto(?)
    )
    SELECT
        COUNT(*) AS total_orders,
        SUM(
            CASE WHEN order_status = 'delivered'
            THEN 1 ELSE 0 END
        ) AS delivered_orders,
        SUM(
            CASE WHEN order_status = 'delivered'
                  AND order_delivered_customer_date IS NOT NULL
                  AND order_estimated_delivery_date IS NOT NULL
            THEN 1 ELSE 0 END
        ) AS usable_delivery_orders,
        SUM(
            CASE WHEN r.order_id IS NOT NULL
            THEN 1 ELSE 0 END
        ) AS orders_with_reviews,
        SUM(
            CASE WHEN order_status = 'delivered'
                  AND order_delivered_customer_date IS NOT NULL
                  AND order_estimated_delivery_date IS NOT NULL
                  AND r.order_id IS NOT NULL
            THEN 1 ELSE 0 END
        ) AS core_analysis_orders
    FROM read_csv_auto(?) AS o
    LEFT JOIN reviewed_orders AS r
        ON o.order_id = r.order_id
    """,
    [
        reviews_file.as_posix(),
        orders_file.as_posix()
    ]
).df()

analysis_coverage

,total_orders,delivered_orders,usable_delivery_orders,orders_with_reviews,core_analysis_orders
0,99441,96478.0,96470.0,98673.0,95824.0


## Initial audit findings

- All 9 expected CSV files are available.
- The dataset contains 99,441 orders.
- 96,478 orders were marked as delivered.
- 96,470 delivered orders have the dates required to calculate delivery delay.
- 98,673 orders can be matched to at least one customer review.
- 95,824 orders satisfy the initial requirements for the core analysis.

### Missing data

- Review titles and written comments are frequently missing because they are optional.
- Missing delivery dates mainly occur among orders that were not completed.
- A small proportion of products have no category information.
- Missing product dimensions are negligible.

### Main table relationships

- `orders` joins to `reviews` through `order_id`.
- `orders` joins to `customers` through `customer_id`.
- `orders` joins to `order_items` through `order_id`.
- `order_items` joins to `products` through `product_id`.
- `order_items` joins to `sellers` through `seller_id`.
- `products` joins to the category translation table through `product_category_name`.

### Initial analysis sample

The initial core sample will retain delivered orders with:

- an actual delivery date;
- an estimated delivery date;
- a customer review.

A separate statistical audit will later examine duplicate records, date consistency, outliers, delay distributions and review-score differences.